# PlurVA zh/id/si: SFT (continue-or-fresh) + GRPO + test predictions

Colab-ready, self-contained notebook version of the pipeline in
`scripts/train_sft_then_grpo.py`, plus a final section that generates the
shared-task submission file (`scripts/predict_test.py`'s job), all in one
run. Nothing in this notebook imports from the local repo's `scripts/`
directory -- every function is inlined below, since a fresh Colab VM won't
have it on `sys.path`. You still need the repo's **data** (and optionally an
existing **adapter** checkpoint) accessible from Colab; see the Drive-mount
cell below.

**Pipeline, in order:**
1. **SFT** (simultaneous zh/id/si, macro-averaged loss) -- either a fresh
   LoRA, or continuing from an existing adapter (e.g. to pick up the
   Indonesian tie-duplication fix in the training data without spending the
   full budget again). Skippable.
2. **GRPO** on top of whatever SFT produced (or the resumed checkpoint if
   SFT was skipped) -- group-relative policy-gradient fine-tuning, reward =
   correctness against the gold letter (or either tied letter for
   Indonesian's ambiguous-vote rows). No KL-to-reference term (would need a
   second 4B-param model copy in memory); add one back if the policy drifts.
3. **Test predictions** -- reloads the best GRPO checkpoint and generates
   `results/test_submission.jsonl` in the shared-task's expected format:
   `{"dataset": "chinese"|"indonesian"|"sri_lankan", "id": <ID>, "LLM_Output": "A"|"B"|"C"|"D"|"Both"|"0"}`,
   one JSON object per line, all three languages in a single file.

Runs on CUDA (Colab GPU runtime), MPS, or CPU, auto-detected.

## Setup

**Before running:** Runtime -> Change runtime type -> GPU (T4/A100/etc).

This notebook expects the repo's `data/` directory (and, if you're
continuing training, an `adapters/.../best` checkpoint) to be reachable from
Colab. The simplest way: sync/upload the whole `PlurvaLLM-SharedTask` repo
to Google Drive, then point `REPO_ROOT` below at it. If you'd rather upload
just the `data/` (and `adapters/`) folders directly into the Colab VM
instead of using Drive, skip the mount cell and set `REPO_ROOT` to wherever
you uploaded them (e.g. `/content/PlurvaLLM-SharedTask`).

In [ ]:
%%capture
!pip install -U transformers peft accelerate tqdm safetensors sentencepiece huggingface_hub


In [ ]:
from pathlib import Path

try:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_ROOT = Path("/content/drive/MyDrive/PlurvaLLM-SharedTask")  # <-- adjust to your Drive path
except ImportError:
    # Not running in Colab (or Drive not wanted) -- point this at wherever
    # data/ and adapters/ actually live.
    REPO_ROOT = Path("/content/PlurvaLLM-SharedTask")

assert (REPO_ROOT / "data").exists(), (
    f"{REPO_ROOT / 'data'} not found -- fix REPO_ROOT above to point at the repo "
    "(or the folder containing data/ and adapters/)."
)
print("REPO_ROOT =", REPO_ROOT)


## Imports

In [ ]:
import json
import os
import random
import re
import shutil
import time
from collections import Counter

# Must be set before torch touches the MPS backend (harmless on Colab's CUDA
# runtime, but keeps this notebook portable if run locally on Apple Silicon).
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import torch
import torch.nn.functional as F
from peft import LoraConfig, PeftModel, get_peft_model
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

LANGS = ["zh", "id", "si"]


## Ported from `scripts/eval_baseline.py`

Prompt building, gold-answer resolution (including the tie-duplication
logic: Indonesian rows with no strict-majority annotator vote resolve to
*both* tied letters as valid candidates instead of one arbitrary pick), and
answer-letter extraction. Both the SFT and GRPO phases below, and the final
test-prediction section, all route through these same functions so prompting
stays identical everywhere.

In [ ]:
DATA_DIR = REPO_ROOT / "data"
LANG_FILES = {
    "zh": DATA_DIR / "chinese_dev.jsonl",
    "id": DATA_DIR / "indonesian_dev.jsonl",
    "si": DATA_DIR / "sri_lankan_dev.jsonl",
}

LETTER_RE = re.compile(r"\b([ABCD])\b")
THINK_RE = re.compile(r"^.*?</think>", re.DOTALL)

SI_FIXED_OPTION_C = "\u0db4\u0dd2\u0dc5\u0dd2\u0dad\u0dd4\u0dbb\u0dd4 \u0daf\u0dd9\u0d9a\u0db8 \u0db1\u0dd2\u0dc0\u0dd0\u0dbb\u0daf\u0dd2\u0dba\u0dd2."
SI_FIXED_OPTION_D = "\u0db4\u0dd2\u0dc5\u0dd2\u0dad\u0dd4\u0dbb\u0dd4 \u0daf\u0dd9\u0d9a\u0db8 \u0db1\u0dd2\u0dc0\u0dd0\u0dbb\u0daf\u0dd2 \u0db1\u0ddc\u0dc0\u0dda."

PROMPT_TEMPLATES = {
    "zh": """You are a Simplified Chinese Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Chinese context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
    "id": """You are an Indonesian Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Indonesian context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
    "si": """You are a Sinhala Expert for answering multiple-choice questions.
    You need to always evaluate these questions based on the Sri Lankan context and provide the most accurate answer.
    You should only respond with the correct option text, without any additional explanation or commentary.
{scenario_line}Question: {question}
Option A: {option_a}
Option B: {option_b}
Option C: {option_c}
Option D: {option_d}
Return only the correct option text. Expected output is either the text of 'A', 'B', 'C', or 'D'.""",
}

SI_GOLD_MAP = {"Both": "C", "0": "D"}


def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def load_rows(lang):
    return read_jsonl(LANG_FILES[lang])


def resolve_gold_candidates(lang, gold_answer):
    """Gold letter(s) as a list: one item normally, or the tied top letters
    (e.g. ["A", "D"]) when Indonesian's 5-annotator vote has no strict
    majority -- either tied letter counts as correct."""
    gold_answer = gold_answer.strip()
    if lang == "si" and gold_answer in SI_GOLD_MAP:
        return [SI_GOLD_MAP[gold_answer]]
    if "," in gold_answer:
        votes = [v.strip() for v in gold_answer.split(",")]
        counts = Counter(votes)
        top_n = counts.most_common(1)[0][1]
        return sorted(letter for letter, n in counts.items() if n == top_n)
    return [gold_answer]


def resolve_gold(lang, gold_answer):
    """Strict single-answer version (used for eval/scoring, not training):
    None if there's no strict majority."""
    candidates = resolve_gold_candidates(lang, gold_answer)
    return candidates[0] if len(candidates) == 1 else None


def resolve_options(lang, row):
    options = {
        "A": row.get("Option_A", ""),
        "B": row.get("Option_B", ""),
        "C": row.get("Option_C", ""),
        "D": row.get("Option_D", ""),
    }
    if lang == "si":
        options["C"] = SI_FIXED_OPTION_C
        options["D"] = SI_FIXED_OPTION_D
    return options


def build_prompt(lang, row, options):
    scenario = row.get("Scenario", "").strip()
    scenario_line = f"Scenario: {scenario}\n" if scenario else ""
    return PROMPT_TEMPLATES[lang].format(
        scenario_line=scenario_line,
        question=row["Question"],
        option_a=options["A"],
        option_b=options["B"],
        option_c=options["C"],
        option_d=options["D"],
    )


def strip_thinking(text):
    return THINK_RE.sub("", text, count=1).strip()


def extract_letter(text, valid_letters, options=None):
    text = strip_thinking(text)
    for m in LETTER_RE.finditer(text):
        if m.group(1) in valid_letters:
            return m.group(1)
    if options is not None:
        norm = text.strip().lower()
        if norm:
            for letter in valid_letters:
                opt_text = options.get(letter, "").strip().lower()
                if opt_text and (opt_text in norm or norm in opt_text):
                    return letter
    return None


## Ported from `scripts/prepare_training_data.py` (optional regeneration)

Only needed if `data/train/train_<lang>.jsonl` / `data/val/val_<lang>.jsonl`
aren't already present in your synced repo copy (they should be, if you
synced the whole repo -- this is here for robustness / in case you only
uploaded the raw `data/*_dev.jsonl` files). Builds the same tie-duplication
fix as the local repo: Indonesian rows with no strict-majority vote produce
**two** training examples (same prompt, one per tied letter) instead of
being dropped, with duplicate pairs kept together on the same side of the
train/val split.

In [ ]:
REGENERATE_SPLITS = False  # set True to (re)build data/train, data/val from data/*_dev.jsonl
SPLIT_SEED = 42
VAL_FRACTION = 0.15


def build_example_groups(lang):
    rows = load_rows(lang)
    groups = []
    tied_rows = 0
    for row in rows:
        candidates = resolve_gold_candidates(lang, row["Gold_Answer"])
        if len(candidates) > 1:
            tied_rows += 1
        options = resolve_options(lang, row)
        prompt = build_prompt(lang, row, options)
        groups.append([{"prompt": prompt, "completion": f" {letter}"} for letter in candidates])
    return groups, tied_rows


if REGENERATE_SPLITS:
    train_dir = REPO_ROOT / "data" / "train"
    val_dir = REPO_ROOT / "data" / "val"
    train_dir.mkdir(parents=True, exist_ok=True)
    val_dir.mkdir(parents=True, exist_ok=True)

    split_rng = random.Random(SPLIT_SEED)
    for lang in LANGS:
        groups, tied_rows = build_example_groups(lang)
        split_rng.shuffle(groups)
        n_val_groups = max(1, int(len(groups) * VAL_FRACTION))
        val_groups = groups[:n_val_groups]
        train_groups = groups[n_val_groups:]

        train_examples = [ex for group in train_groups for ex in group]
        val_examples = [ex for group in val_groups for ex in group]

        with open(train_dir / f"train_{lang}.jsonl", "w", encoding="utf-8") as f:
            for ex in train_examples:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")
        with open(val_dir / f"val_{lang}.jsonl", "w", encoding="utf-8") as f:
            for ex in val_examples:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")

        print(f"{lang}: {len(train_examples)} train examples ({len(train_groups)} rows), "
              f"{len(val_examples)} val examples ({len(val_groups)} rows), "
              f"{tied_rows} tied rows duplicated")
else:
    print("REGENERATE_SPLITS is False -- using data/train and data/val as already synced.")


## Ported from `scripts/train_macro_lora_pt.py` (SFT mechanics)

Prompt-masked next-token loss, sampled with replacement per language so
smaller datasets upsample to fill an equal-size slice every step, and the
EQUAL-weight macro average across languages via three separate `backward()`
calls (frees each language's compute graph before starting the next --
matters given how memory-hungry Qwen3.5's linear-attention fallback path
already is).

In [ ]:
class PromptCompletionDataset:
    def __init__(self, rows, tokenizer, max_seq_length):
        self.examples = []
        for row in rows:
            messages = [{"role": "user", "content": row["prompt"]}]
            try:
                chat_prompt = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False,
                    enable_thinking=False,
                )
            except TypeError:
                chat_prompt = tokenizer.apply_chat_template(
                    messages, add_generation_prompt=True, tokenize=False,
                )
            prompt_ids = tokenizer(chat_prompt, add_special_tokens=False)["input_ids"]
            completion_ids = tokenizer(row["completion"], add_special_tokens=False)["input_ids"]
            input_ids = prompt_ids + completion_ids
            if len(input_ids) > max_seq_length:
                input_ids = input_ids[-max_seq_length:]
                prompt_len = max(0, len(input_ids) - len(completion_ids))
            else:
                prompt_len = len(prompt_ids)
            labels = [-100] * prompt_len + input_ids[prompt_len:]
            self.examples.append({"input_ids": input_ids, "labels": labels})

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        return self.examples[idx]


def sample_batch(dataset, batch_size, rng):
    idxs = [rng.randrange(len(dataset)) for _ in range(batch_size)]
    return [dataset[i] for i in idxs]


def collate(examples, pad_token_id, device):
    max_len = max(len(e["input_ids"]) for e in examples)
    input_ids = torch.full((len(examples), max_len), pad_token_id, dtype=torch.long)
    labels = torch.full((len(examples), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(examples), max_len), dtype=torch.long)
    for i, e in enumerate(examples):
        L = len(e["input_ids"])
        input_ids[i, :L] = torch.tensor(e["input_ids"], dtype=torch.long)
        labels[i, :L] = torch.tensor(e["labels"], dtype=torch.long)
        attention_mask[i, :L] = 1
    return {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device),
        "labels": labels.to(device),
    }


def _language_batch_loss(model, examples, pad_token_id, device):
    batch = collate(examples, pad_token_id, device)
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = out.logits[:, :-1, :]
    labels = batch["labels"][:, 1:]

    token_ce = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
        ignore_index=-100, reduction="none",
    ).view(labels.shape)
    valid = (labels != -100)
    tok_count = valid.sum(dim=1).clamp(min=1)
    per_example_loss = (token_ce * valid).sum(dim=1) / tok_count
    with torch.no_grad():
        correct = (logits.argmax(dim=-1) == labels) & valid
        accuracy = (correct.sum().float() / valid.sum().clamp(min=1)).item()
    return per_example_loss.mean(), accuracy


def macro_lang_backward(model, batches_by_lang, pad_token_id, device):
    total = 0.0
    lang_acc = {}
    for lang in LANGS:
        lang_loss, acc = _language_batch_loss(model, batches_by_lang[lang], pad_token_id, device)
        (lang_loss / len(LANGS)).backward()
        total += lang_loss.item()
        lang_acc[lang] = acc
    return total / len(LANGS), lang_acc


@torch.no_grad()
def per_language_validate(model, val_datasets, batch_size, val_batches, pad_token_id, device, rng):
    model.eval()
    lang_losses = {}
    for lang, ds in val_datasets.items():
        losses = []
        for _ in range(val_batches):
            batch = collate(sample_batch(ds, batch_size, rng), pad_token_id, device)
            out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
            logits = out.logits[:, :-1, :]
            labels = batch["labels"][:, 1:]
            token_ce = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
                ignore_index=-100, reduction="none",
            ).view(labels.shape)
            valid = (labels != -100)
            tok_count = valid.sum(dim=1).clamp(min=1)
            per_example_loss = (token_ce * valid).sum(dim=1) / tok_count
            losses.append(per_example_loss.mean().item())
        lang_losses[lang] = sum(losses) / len(losses)
        tqdm.write(f"  val_loss[{lang}]={lang_losses[lang]:.4f}")
    model.train()
    return sum(lang_losses.values()) / len(lang_losses)


def save_best_known(model, adapter_path, best_adapter_path):
    if best_adapter_path.exists():
        shutil.copytree(best_adapter_path, adapter_path, dirs_exist_ok=True)
    else:
        model.save_pretrained(str(adapter_path))


## Ported from `scripts/train_sft_then_grpo.py` (GRPO mechanics)

For each sampled row: sample a group of completions at temperature > 0,
reward each by correctness (reusing `resolve_gold_candidates`), normalize
rewards within the group to advantages, skip degenerate (zero-variance)
groups, then a single combined weighted-CE-loss backward per step.

In [ ]:
def load_split_rows(lang, seed, val_fraction):
    rows = load_rows(lang)
    rng = random.Random(seed)
    order = list(range(len(rows)))
    rng.shuffle(order)
    n_val = max(1, int(len(rows) * val_fraction))
    val_idx = set(order[:n_val])
    train_rows = [rows[i] for i in order if i not in val_idx]
    val_rows = [rows[i] for i in order if i in val_idx]
    return train_rows, val_rows


def build_chat_prompt(tokenizer, lang, row, options):
    prompt = build_prompt(lang, row, options)
    messages = [{"role": "user", "content": prompt}]
    try:
        chat_prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, enable_thinking=False,
        )
    except TypeError:
        chat_prompt = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False,
        )
    return prompt, chat_prompt


def reward_fn(pred_letter, gold_candidates, unparseable_penalty):
    if pred_letter is None:
        return unparseable_penalty
    return 1.0 if pred_letter in gold_candidates else 0.0


@torch.no_grad()
def sample_group(model, tokenizer, device, lang, row, group_size, max_new_tokens, unparseable_penalty,
                  do_sample=True, temperature=None, top_p=None):
    options = resolve_options(lang, row)
    valid_letters = {l for l in "ABCD" if options[l]}
    gold_candidates = set(resolve_gold_candidates(lang, row["Gold_Answer"]))
    _, chat_prompt = build_chat_prompt(tokenizer, lang, row, options)

    inputs = tokenizer(chat_prompt, return_tensors="pt", add_special_tokens=False).to(device)
    prompt_len = inputs["input_ids"].shape[1]

    gen_kwargs = dict(
        max_new_tokens=max_new_tokens,
        pad_token_id=tokenizer.pad_token_id,
        num_return_sequences=group_size,
    )
    if do_sample:
        gen_kwargs.update(do_sample=True, temperature=temperature, top_p=top_p)
    else:
        gen_kwargs.update(do_sample=False)

    output_ids = model.generate(**inputs, **gen_kwargs)

    samples = []
    for seq in output_ids:
        completion_ids = seq[prompt_len:].tolist()
        while completion_ids and completion_ids[-1] == tokenizer.pad_token_id:
            completion_ids.pop()
        response = tokenizer.decode(completion_ids, skip_special_tokens=True)
        pred = extract_letter(response, valid_letters, options)
        reward = reward_fn(pred, gold_candidates, unparseable_penalty)
        samples.append({
            "input_ids": inputs["input_ids"][0].tolist() + completion_ids,
            "prompt_len": prompt_len,
            "reward": reward,
        })
    return samples


def collate_weighted(samples, pad_token_id, device):
    max_len = max(len(s["input_ids"]) for s in samples)
    input_ids = torch.full((len(samples), max_len), pad_token_id, dtype=torch.long)
    labels = torch.full((len(samples), max_len), -100, dtype=torch.long)
    attention_mask = torch.zeros((len(samples), max_len), dtype=torch.long)
    weights = torch.zeros(len(samples), dtype=torch.float)
    for i, s in enumerate(samples):
        ids = s["input_ids"]
        L = len(ids)
        input_ids[i, :L] = torch.tensor(ids, dtype=torch.long)
        attention_mask[i, :L] = 1
        labels[i, s["prompt_len"]:L] = torch.tensor(ids[s["prompt_len"]:], dtype=torch.long)
        weights[i] = s["advantage"]
    return {
        "input_ids": input_ids.to(device),
        "attention_mask": attention_mask.to(device),
        "labels": labels.to(device),
        "weights": weights.to(device),
    }


def grpo_step(model, tokenizer, device, batches_by_lang, group_size, temperature, top_p,
              max_new_tokens, unparseable_penalty):
    all_samples = []
    reward_log = {}
    for lang, rows in batches_by_lang.items():
        lang_rewards = []
        for row in rows:
            group = sample_group(
                model, tokenizer, device, lang, row, group_size, max_new_tokens, unparseable_penalty,
                do_sample=True, temperature=temperature, top_p=top_p,
            )
            rewards = torch.tensor([s["reward"] for s in group])
            lang_rewards.extend(rewards.tolist())
            std, mean = rewards.std(unbiased=False), rewards.mean()
            if std < 1e-6:
                continue  # degenerate group: every sample scored the same, no signal
            advantages = (rewards - mean) / (std + 1e-6)
            for s, adv in zip(group, advantages.tolist()):
                s["advantage"] = adv
                all_samples.append(s)
        reward_log[lang] = sum(lang_rewards) / len(lang_rewards) if lang_rewards else float("nan")

    if not all_samples:
        return None, reward_log

    batch = collate_weighted(all_samples, tokenizer.pad_token_id, device)
    out = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"])
    logits = out.logits[:, :-1, :]
    labels = batch["labels"][:, 1:]

    token_ce = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), labels.reshape(-1),
        ignore_index=-100, reduction="none",
    ).view(labels.shape)
    valid = (labels != -100)
    tok_count = valid.sum(dim=1).clamp(min=1)
    per_example_loss = (token_ce * valid).sum(dim=1) / tok_count
    weighted_loss = (per_example_loss * batch["weights"]).mean()

    weighted_loss.backward()
    return weighted_loss.item(), reward_log


@torch.no_grad()
def grpo_validate(model, tokenizer, device, val_rows_by_lang, rng, n_rows_per_lang, max_new_tokens,
                   unparseable_penalty):
    model.eval()
    lang_acc = {}
    for lang in LANGS:
        rows = rng.sample(val_rows_by_lang[lang], min(n_rows_per_lang, len(val_rows_by_lang[lang])))
        correct = 0
        for row in rows:
            sample = sample_group(
                model, tokenizer, device, lang, row, group_size=1, max_new_tokens=max_new_tokens,
                unparseable_penalty=unparseable_penalty, do_sample=False,
            )[0]
            correct += 1 if sample["reward"] == 1.0 else 0
        lang_acc[lang] = correct / len(rows)
    model.train()
    return lang_acc, sum(lang_acc.values()) / len(lang_acc)


## Configuration

In [ ]:
MODEL_ID = "Qwen/Qwen3.5-4B"
SEED = 42
MAX_SEQ_LENGTH = 768
LORA_RANK = 16
LORA_ALPHA = 32.0
GRAD_CHECKPOINT = True

RESUME_SFT_ADAPTER = str(REPO_ROOT / "adapters" / "macro_lora_pt" / "best")  # None for a fresh LoRA
SKIP_SFT = False  # True runs GRPO directly on RESUME_SFT_ADAPTER (which must then be set)

# --- SFT ---
TRAIN_DIR = REPO_ROOT / "data" / "train"
VAL_DIR = REPO_ROOT / "data" / "val"
SFT_ADAPTER_PATH = REPO_ROOT / "adapters" / "macro_lora_pt"
SFT_PER_LANG_BATCH_SIZE = 2
SFT_ITERS = 500
SFT_STEPS_PER_REPORT = 10
SFT_STEPS_PER_EVAL = 5
SFT_VAL_BATCHES = 10
SFT_PATIENCE = 5
SFT_LEARNING_RATE = 1e-4

# --- GRPO ---
GRPO_ADAPTER_PATH = REPO_ROOT / "adapters" / "macro_lora_grpo"
GRPO_VAL_FRACTION = 0.15
GRPO_ROWS_PER_LANG_PER_STEP = 1
GRPO_GROUP_SIZE = 8
GRPO_TEMPERATURE = 0.8
GRPO_TOP_P = 0.95
GRPO_MAX_NEW_TOKENS = 40
GRPO_UNPARSEABLE_PENALTY = -0.2
GRPO_ITERS = 200
GRPO_LEARNING_RATE = 5e-6
GRPO_STEPS_PER_EVAL = 10
GRPO_VAL_ROWS_PER_LANG = 10
GRPO_STEPS_PER_SAVE = 20
GRPO_PATIENCE = 5

if SKIP_SFT:
    assert RESUME_SFT_ADAPTER, "SKIP_SFT requires RESUME_SFT_ADAPTER (GRPO needs a starting checkpoint)"


## Load model + tokenizer + adapter

In [ ]:
if torch.cuda.is_available():
    device = "cuda"
elif torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

if device == "cuda":
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
elif device == "mps":
    dtype = torch.float16
else:
    dtype = torch.bfloat16

print(f"Loading {MODEL_ID} on {device} ({dtype}) ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype)

if RESUME_SFT_ADAPTER:
    print(f"Loading trainable LoRA adapter from {RESUME_SFT_ADAPTER} ...")
    model = PeftModel.from_pretrained(base_model, RESUME_SFT_ADAPTER, is_trainable=True)
else:
    lora_config = LoraConfig(
        r=LORA_RANK, lora_alpha=LORA_ALPHA, lora_dropout=0.0,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                         "gate_proj", "up_proj", "down_proj"],
        task_type="CAUSAL_LM",
    )
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

if GRAD_CHECKPOINT:
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    print("Gradient checkpointing enabled (trades compute for memory).")
model.to(device)
model.train()


## Run SFT

In [ ]:
def run_sft():
    train_datasets = {
        lang: PromptCompletionDataset(
            read_jsonl(TRAIN_DIR / f"train_{lang}.jsonl"), tokenizer, MAX_SEQ_LENGTH
        )
        for lang in LANGS
    }
    val_datasets = {
        lang: PromptCompletionDataset(
            read_jsonl(VAL_DIR / f"val_{lang}.jsonl"), tokenizer, MAX_SEQ_LENGTH
        )
        for lang in LANGS
    }

    optimizer = torch.optim.AdamW(model.parameters(), lr=SFT_LEARNING_RATE)
    rng = random.Random(SEED)

    adapter_path = Path(SFT_ADAPTER_PATH)
    adapter_path.mkdir(parents=True, exist_ok=True)
    best_adapter_path = adapter_path / "best"
    best_val_loss = float("inf")
    stale_evals = 0

    losses, steps, t0 = 0.0, 0, time.time()
    lang_acc_sums = {lang: 0.0 for lang in LANGS}
    print("[sft] Starting (simultaneous zh/id/si, macro-averaged loss)...")
    pbar = tqdm(range(1, SFT_ITERS + 1), desc="sft", unit="it")
    for it in pbar:
        step_t0 = time.time()
        batches_by_lang = {
            lang: sample_batch(train_datasets[lang], SFT_PER_LANG_BATCH_SIZE, rng)
            for lang in LANGS
        }
        optimizer.zero_grad()
        loss_value, lang_acc = macro_lang_backward(model, batches_by_lang, tokenizer.pad_token_id, device)
        optimizer.step()
        step_dt = time.time() - step_t0

        losses += loss_value
        steps += 1
        for lang in LANGS:
            lang_acc_sums[lang] += lang_acc[lang]
        pbar.set_postfix(loss=f"{loss_value:.4f}")

        acc_str = " ".join(f"acc[{lang}]={lang_acc[lang]:.3f}" for lang in LANGS)
        tqdm.write(f"[sft {it}/{SFT_ITERS}] loss={loss_value:.4f} {acc_str} dt={step_dt:.1f}s")

        if it % SFT_STEPS_PER_REPORT == 0 or it == SFT_ITERS:
            elapsed = time.time() - t0
            acc_str = " ".join(f"acc[{lang}]={lang_acc_sums[lang] / steps:.3f}" for lang in LANGS)
            tqdm.write(f"[sft {it}] train_loss(macro)={losses / steps:.4f} {acc_str} elapsed={elapsed:.0f}s")
            losses, steps = 0.0, 0
            lang_acc_sums = {lang: 0.0 for lang in LANGS}

        if it % SFT_STEPS_PER_EVAL == 0 or it == SFT_ITERS:
            val_loss = per_language_validate(
                model, val_datasets, SFT_PER_LANG_BATCH_SIZE, SFT_VAL_BATCHES,
                tokenizer.pad_token_id, device, rng,
            )
            tqdm.write(f"[sft {it}] val_loss(macro)={val_loss:.4f}")
            stop_early = False
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                stale_evals = 0
                model.save_pretrained(str(best_adapter_path))
                tqdm.write(f"[sft {it}] New best macro val loss; saved best adapter to {best_adapter_path}")
            else:
                stale_evals += 1
                tqdm.write(f"[sft {it}] No improvement ({stale_evals}/{SFT_PATIENCE})")
                if SFT_PATIENCE > 0 and stale_evals >= SFT_PATIENCE:
                    tqdm.write(f"[sft {it}] Early stopping: no macro val loss improvement for {SFT_PATIENCE} evals.")
                    stop_early = True
            save_best_known(model, adapter_path, best_adapter_path)
            tqdm.write(f"[sft {it}] Saved best-known LoRA adapter to {adapter_path}")
            if stop_early:
                break

    save_best_known(model, adapter_path, best_adapter_path)
    print(f"[sft] Saved final (best-known) LoRA adapter to {adapter_path}")


if SKIP_SFT:
    print("SKIP_SFT is True: skipping SFT phase, running GRPO directly on the loaded adapter.")
else:
    run_sft()


## Run GRPO

In [ ]:
def run_grpo():
    train_rows_by_lang = {}
    val_rows_by_lang = {}
    for lang in LANGS:
        train_rows_by_lang[lang], val_rows_by_lang[lang] = load_split_rows(
            lang, SEED, GRPO_VAL_FRACTION
        )
        print(f"[grpo] {lang}: {len(train_rows_by_lang[lang])} train rows, {len(val_rows_by_lang[lang])} val rows")

    out_adapter_path = Path(GRPO_ADAPTER_PATH)
    out_adapter_path.mkdir(parents=True, exist_ok=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=GRPO_LEARNING_RATE)
    rng = random.Random(SEED)

    best_val_macro = -1.0
    stale_evals = 0
    t0 = time.time()
    print("[grpo] Starting...")
    pbar = tqdm(range(1, GRPO_ITERS + 1), desc="grpo", unit="it")
    for it in pbar:
        stop_early = False
        batches_by_lang = {
            lang: [rng.choice(train_rows_by_lang[lang]) for _ in range(GRPO_ROWS_PER_LANG_PER_STEP)]
            for lang in LANGS
        }

        optimizer.zero_grad()
        loss_value, reward_log = grpo_step(
            model, tokenizer, device, batches_by_lang, GRPO_GROUP_SIZE, GRPO_TEMPERATURE,
            GRPO_TOP_P, GRPO_MAX_NEW_TOKENS, GRPO_UNPARSEABLE_PENALTY,
        )
        if loss_value is not None:
            optimizer.step()

        reward_str = " ".join(f"r[{lang}]={reward_log[lang]:.2f}" for lang in LANGS)
        pbar.set_postfix(loss=f"{loss_value:.4f}" if loss_value is not None else "skip")
        loss_display = loss_value if loss_value is not None else float("nan")
        tqdm.write(f"[grpo {it}/{GRPO_ITERS}] loss={loss_display:.4f} {reward_str}")

        if it % GRPO_STEPS_PER_EVAL == 0 or it == GRPO_ITERS:
            lang_acc, val_macro = grpo_validate(
                model, tokenizer, device, val_rows_by_lang, rng, GRPO_VAL_ROWS_PER_LANG,
                GRPO_MAX_NEW_TOKENS, GRPO_UNPARSEABLE_PENALTY,
            )
            acc_str = " ".join(f"val_acc[{lang}]={lang_acc[lang]:.3f}" for lang in LANGS)
            tqdm.write(f"[grpo {it}] val_macro_acc={val_macro:.4f} {acc_str}")
            if val_macro > best_val_macro:
                best_val_macro = val_macro
                stale_evals = 0
                model.save_pretrained(str(out_adapter_path / "best"))
                tqdm.write(f"[grpo {it}] New best val macro acc; saved adapter to {out_adapter_path / 'best'}")
            else:
                stale_evals += 1
                tqdm.write(f"[grpo {it}] No improvement ({stale_evals}/{GRPO_PATIENCE})")
                if GRPO_PATIENCE > 0 and stale_evals >= GRPO_PATIENCE:
                    tqdm.write(f"[grpo {it}] Early stopping: no val macro-acc improvement for {GRPO_PATIENCE} evals.")
                    stop_early = True

        if it % GRPO_STEPS_PER_SAVE == 0 or it == GRPO_ITERS:
            model.save_pretrained(str(out_adapter_path))
            tqdm.write(f"[grpo {it}] Saved latest adapter to {out_adapter_path}")

        if stop_early:
            break

    elapsed = time.time() - t0
    print(f"[grpo] Done in {elapsed:.0f}s. Best val macro acc: {best_val_macro:.4f}. "
          f"Best adapter: {out_adapter_path / 'best'}, latest: {out_adapter_path}")


run_grpo()


## Test predictions (ported from `scripts/predict_test.py`)

Reloads the **best** GRPO checkpoint from disk (not just whatever's left in
memory after the last training step -- early stopping or a late regression
could mean the in-memory weights aren't the best-scoring ones) and generates
predictions for the unlabeled shared-task test sets
(`data/test/*_test_without_gold.jsonl`, no `Gold_Answer` field, so this is
prediction only, no scoring).

**Output format** -- one JSON object per line, all three languages in a
single file: `{"dataset": "chinese"|"indonesian"|"sri_lankan", "id": <ID>, "LLM_Output": "A"|"B"|"C"|"D"|"Both"|"0"}`.
Sri Lankan's internal `C`/`D` meta-answers get mapped back to `"Both"`/`"0"`
for submission.

In [ ]:
PREDICT_ADAPTER_PATH = str(GRPO_ADAPTER_PATH / "best")  # reload the best checkpoint, not the live model
PREDICT_MAX_TOKENS = 40
PREDICT_OUT = REPO_ROOT / "results" / "test_submission.jsonl"

TEST_DATA_DIR = REPO_ROOT / "data" / "test"
TEST_FILES = {
    "zh": TEST_DATA_DIR / "chinese_test_without_gold.jsonl",
    "id": TEST_DATA_DIR / "indonesian_test_without_gold.jsonl",
    "si": TEST_DATA_DIR / "sri_lankan_test_without_gold.jsonl",
}
DATASET_NAMES = {"zh": "chinese", "id": "indonesian", "si": "sri_lankan"}
SI_OUTPUT_MAP = {"C": "Both", "D": "0"}


def load_test_rows(lang):
    rows = []
    with open(TEST_FILES[lang], encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def to_submission_output(lang, letter):
    if lang == "si":
        return SI_OUTPUT_MAP.get(letter, letter)
    return letter


print(f"Reloading best checkpoint from {PREDICT_ADAPTER_PATH} for prediction ...")
predict_base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=dtype)
predict_model = PeftModel.from_pretrained(predict_base_model, PREDICT_ADAPTER_PATH)
predict_model.to(device)
predict_model.eval()

PREDICT_OUT.parent.mkdir(parents=True, exist_ok=True)
out_f = open(PREDICT_OUT, "w", encoding="utf-8")

total = 0
unparseable = 0
t0 = time.time()

for lang in LANGS:
    rows = load_test_rows(lang)
    for row in tqdm(rows, desc=f"Predicting {lang}"):
        options = resolve_options(lang, row)
        valid_letters = {l for l in "ABCD" if options[l]}
        prompt = build_prompt(lang, row, options)

        messages = [{"role": "user", "content": prompt}]
        try:
            chat_prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False, enable_thinking=False,
            )
        except TypeError:
            chat_prompt = tokenizer.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=False,
            )

        inputs = tokenizer(chat_prompt, return_tensors="pt", add_special_tokens=False).to(device)
        with torch.no_grad():
            output_ids = predict_model.generate(
                **inputs,
                max_new_tokens=PREDICT_MAX_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
            )
        response = tokenizer.decode(
            output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
        )

        pred = extract_letter(response, valid_letters, options)
        total += 1
        if pred is None:
            unparseable += 1
            pred = "A"  # submission format has no "unknown" value

        out_f.write(
            json.dumps(
                {
                    "dataset": DATASET_NAMES[lang],
                    "id": row["ID"],
                    "LLM_Output": to_submission_output(lang, pred),
                },
                ensure_ascii=False,
            )
            + "\n"
        )
        out_f.flush()

out_f.close()
elapsed = time.time() - t0
print(
    f"Done: {total} rows written to {PREDICT_OUT}, "
    f"{unparseable} unparseable (defaulted to 'A'), elapsed={elapsed:.0f}s"
)


## (Optional) Download the submission file

If `REPO_ROOT` is on Drive, `results/test_submission.jsonl` is already
saved there. If you'd rather grab it directly from the Colab session:

In [ ]:
try:
    from google.colab import files
    files.download(str(PREDICT_OUT))
except ImportError:
    print(f"Not in Colab -- file is at {PREDICT_OUT}")
